# Sequential Accelerometer Readings - Temporal Features

## Overview
This notebook demonstrates temporal feature engineering for accelerometer data and how a simple model captures temporal dependencies using sequential patterns.

In [11]:
import pandas as pd
import numpy as np
import glob
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
import lightgbm as lgb
import seaborn as sns
import warnings
import os
warnings.filterwarnings('ignore')

## 1. Load and Prepare Data

In [12]:
TRAIN_FILE_NUMBERS = 11020
TRAIN_USER_NUMBERS = 60
TEST_FILE_NUMBERS = 6849
TEST_USER_NUMBERS = 40

def file_csv_to_df(train, file_id, user_id=None, g_to_ms2=9.81):
    if train:
        assert 1 <= file_id <= TRAIN_FILE_NUMBERS
        folder = "train"
    else:
        assert TRAIN_FILE_NUMBERS + 1 <= file_id <= TRAIN_FILE_NUMBERS + TEST_FILE_NUMBERS
        folder = "test"
    
    if user_id is not None:
        assert 1 <= user_id <= TRAIN_USER_NUMBERS + TEST_USER_NUMBERS
        df = pd.read_csv(f'data/{folder}/User_{user_id:03d}/{file_id:05d}.csv')
    else:
        pattern = f"data/{folder}/User_*/{file_id:05d}.csv"
        matches = glob.glob(pattern)
        if len(matches) == 0:
            raise FileNotFoundError(f"No file found for file_id={file_id}")
        df = pd.read_csv(matches[0])

    df['mean_x'] *= g_to_ms2
    df['mean_y'] *= g_to_ms2
    df['mean_z'] *= g_to_ms2
    df['std_x'] *= g_to_ms2
    df['std_y'] *= g_to_ms2
    df['std_z'] *= g_to_ms2
    return df

print("Data loading functions defined")

Data loading functions defined


## 2. Temporal Features Engineering

Temporal features capture the dynamics of accelerometer data:
- **First-order differences**: Rate of change in acceleration
- **Second-order differences**: Acceleration of acceleration
- **Rolling statistics**: Mean/std over windows
- **Velocity features**: Cumulative acceleration over time
- **Energy features**: Magnitude of motion

In [13]:
def add_temporal_features(df, window_sizes=[3, 5]):
    df = df.copy()
    
    # Acceleration magnitude
    df['accel_magnitude'] = np.sqrt(df['mean_x']**2 + df['mean_y']**2 + df['mean_z']**2)
    
    # First-order differences (rate of change)
    df['delta_mean_x'] = df['mean_x'].diff().fillna(0)
    df['delta_mean_y'] = df['mean_y'].diff().fillna(0)
    df['delta_mean_z'] = df['mean_z'].diff().fillna(0)
    
    # Second-order differences (jerk)
    df['delta2_mean_x'] = df['delta_mean_x'].diff().fillna(0)
    df['delta2_mean_y'] = df['delta_mean_y'].diff().fillna(0)
    df['delta2_mean_z'] = df['delta_mean_z'].diff().fillna(0)
    
    # Cumulative acceleration (velocity approximation)
    df['vel_x'] = df['mean_x'].cumsum()
    df['vel_y'] = df['mean_y'].cumsum()
    df['vel_z'] = df['mean_z'].cumsum()
    df['velocity_magnitude'] = np.sqrt(df['vel_x']**2 + df['vel_y']**2 + df['vel_z']**2)
    
    # Rolling statistics (temporal context)
    for ws in window_sizes:
        df[f'roll_mean_accel_{ws}'] = (
            df['accel_magnitude']
            .rolling(ws, center=True)
            .mean()
            .bfill()
            .ffill()
        )
        df[f'roll_std_accel_{ws}'] = (
            df['accel_magnitude']
            .rolling(ws, center=True)
            .std()
            .bfill()
            .ffill()
        )
        
    # Energy features (motion intensity)
    df['energy'] = df['accel_magnitude'] * np.sqrt(df['std_x']**2 + df['std_y']**2 + df['std_z']**2)
    
    return df

print("Temporal features function defined")

Temporal features function defined


## 3. Sequential Context Features

In [14]:
def add_sequence_context(df, history_steps=3):
    df = df.copy()
    features = ['mean_x', 'mean_y', 'mean_z', 'std_x', 'std_y', 'std_z', 'accel_magnitude']
    for step in range(1, history_steps + 1):
        for feat in features:
            df[f'{feat}_t-{step}'] = (
                df[feat]
                .shift(step)
                .bfill()
            )
    return df

print("Sequence context function defined")

Sequence context function defined


## 4. Load Training Data with Temporal Features

In [15]:
# Check if data exists and try loading
print("Checking for data files...")
train_data_dir = 'data/train'
test_data_dir = 'data/test'

if os.path.exists(train_data_dir):
    user_dirs = glob.glob(f'{train_data_dir}/User_*')
    print(f"Found {len(user_dirs)} user directories")
    if user_dirs:
        sample_files = glob.glob(f'{user_dirs[0]}/*.csv')
        print(f"Sample user has {len(sample_files)} CSV files")
else:
    print(f"WARNING: {train_data_dir} not found.")

# Load training data
X_train_list = []
y_train_list = []
feature_cols = None
error_count = 0
success_count = 0

print("\nLoading training data...")
sample_rate = 1
file_ids = np.random.choice(
    np.arange(1, TRAIN_FILE_NUMBERS + 1),
    size=TRAIN_FILE_NUMBERS,
    replace=False
)

for i, file_id in enumerate(file_ids):
    try:
        df = file_csv_to_df(True, file_id)
        df = add_temporal_features(df)
        df = add_sequence_context(df, history_steps=3)
        
        y_val = df['label'].iloc[0]
        current_feature_cols = [col for col in df.columns if col != 'label']
        if feature_cols is None:
            feature_cols = current_feature_cols
        
        X_train_list.append(df[feature_cols].values)
        y_train_list.append(np.full(len(df), y_val))
        success_count += 1
        if success_count % 10 == 0:
            print(f"  Loaded {success_count} files...")
    except Exception as e:
        error_count += 1
        if error_count == 1:
            print(f"First error loading file {file_id}: {type(e).__name__}: {e}")
        continue

print(f"\nSuccessfully loaded: {success_count} files")
print(f"Failed: {error_count} files")

if len(X_train_list) == 0:
    print("\nNo data files found. Creating synthetic data for demonstration...")
    np.random.seed(42)
    n_samples = 2000
    n_features = 50
    X_train = np.random.randn(n_samples, n_features) * 10
    y_train = np.random.randint(0, 6, n_samples)
    feature_cols = [f'feat_{i}' for i in range(n_features)]
    print(f"Created synthetic training data: {X_train.shape}")
else:
    X_train = np.vstack(X_train_list)
    y_train = np.concatenate(y_train_list)

print(f"\nTraining data shape: {X_train.shape}")
print(f"Features: {len(feature_cols)}")
print(f"Class distribution:")
for label in range(6):
    count = np.sum(y_train == label)
    pct = 100 * count / len(y_train) if len(y_train) > 0 else 0
    print(f"  Label {label}: {count} samples ({pct:.1f}%)")

Checking for data files...
Found 60 user directories
Sample user has 196 CSV files

Loading training data...
  Loaded 10 files...
  Loaded 20 files...
  Loaded 30 files...
  Loaded 40 files...
  Loaded 50 files...
  Loaded 60 files...
  Loaded 70 files...
  Loaded 80 files...
  Loaded 90 files...
  Loaded 100 files...
  Loaded 110 files...
  Loaded 120 files...
  Loaded 130 files...
  Loaded 140 files...
  Loaded 150 files...
  Loaded 160 files...
  Loaded 170 files...
  Loaded 180 files...
  Loaded 190 files...
  Loaded 200 files...
  Loaded 210 files...
  Loaded 220 files...
  Loaded 230 files...
  Loaded 240 files...
  Loaded 250 files...
  Loaded 260 files...
  Loaded 270 files...
  Loaded 280 files...
  Loaded 290 files...
  Loaded 300 files...
  Loaded 310 files...
  Loaded 320 files...
  Loaded 330 files...
  Loaded 340 files...
  Loaded 350 files...
  Loaded 360 files...
  Loaded 370 files...
  Loaded 380 files...
  Loaded 390 files...
  Loaded 400 files...
  Loaded 410 files..

## 5. Train Temporal Model

In [16]:
# Normalize features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)

# Train LightGBM classifier
print('Training LightGBM classifier with temporal features...')
model = lgb.LGBMClassifier(
    n_estimators=100,
    num_leaves=31,
    learning_rate=0.05,
    max_depth=10,
    random_state=42,
    verbose=-1,
    n_jobs=-1
)
model.fit(X_train_scaled, y_train)
print('Training complete!')
print(f'Train accuracy: {model.score(X_train_scaled, y_train):.4f}')

Training LightGBM classifier with temporal features...
Training complete!
Train accuracy: 0.8614


## 6. Evaluate on Test Data

In [17]:
    # Load test data
X_test_list = []
y_test_list = []
test_success = 0

print("Loading test data...")
test_file_ids = list(range(TRAIN_FILE_NUMBERS + 1, min(TRAIN_FILE_NUMBERS + 300 + 1, TRAIN_FILE_NUMBERS + TEST_FILE_NUMBERS + 1), 1))

for i, file_id in enumerate(test_file_ids):
    try:
        df = file_csv_to_df(False, file_id)
        df = add_temporal_features(df)
        df = add_sequence_context(df, history_steps=3)
        X_test_list.append(df[feature_cols].values)
        y_test_list.append(np.full(len(df), y_val))
        test_success += 1
    except Exception as e:
        print(f"Error loading test file {file_id}: {type(e).__name__}: {e}")
        break

print(f"Successfully loaded: {test_success} test files")

if len(X_test_list) == 0:
    print("No test data found. Creating synthetic test data...")
    X_test = np.random.randn(500, X_train.shape[1]) * 10
    y_test = np.random.randint(0, 6, 500)
else:
    X_test = np.vstack(X_test_list)
    y_test = np.concatenate(y_test_list)

X_test_scaled = scaler.transform(X_test)
print(f"\nTest data shape: {X_test.shape}")

Loading test data...
Successfully loaded: 300 test files

Test data shape: (90000, 45)


## 7. Results and Analysis

In [18]:
# Make predictions
y_pred = model.predict(X_test_scaled)

# Calculate metrics
overall_acc = accuracy_score(y_test, y_pred)
print(f"\n{'='*60}")
print(f"TEMPORAL MODEL RESULTS (with sequential features)")
print(f"{'='*60}")
print(f"\nOverall Accuracy: {overall_acc:.4f}")

# Per-class metrics
print(f"\n{'Label':<6} {'Accuracy':<12} {'Precision':<12} {'Recall':<12} {'F1-Score':<12}")
print("-" * 54)

for label in range(6):
    mask = y_test == label
    if np.sum(mask) > 0:
        acc = accuracy_score(y_test[mask], y_pred[mask])
        prec = precision_score(y_test[mask], y_pred[mask], zero_division=0, average='weighted')
        rec = recall_score(y_test[mask], y_pred[mask], zero_division=0, average='weighted')
        f1 = f1_score(y_test[mask], y_pred[mask], zero_division=0, average='weighted')
        print(f"{label:<6} {acc:<12.4f} {prec:<12.4f} {rec:<12.4f} {f1:<12.4f}")


TEMPORAL MODEL RESULTS (with sequential features)

Overall Accuracy: 0.7951

Label  Accuracy     Precision    Recall       F1-Score    
------------------------------------------------------
1      0.7951       1.0000       0.7951       0.8859      


## 8. Feature Importance

In [19]:
# Get feature importances
importances = model.feature_importances_
indices = np.argsort(importances)[::-1][:15]

print("\nTOP 15 MOST IMPORTANT FEATURES:")
print("\nRank  Feature Name                     Importance")
print("-" * 55)

for rank, idx in enumerate(indices, 1):
    feat_name = feature_cols[idx]
    importance = importances[idx]
    is_temporal = 'delta' in feat_name or 't-' in feat_name or 'roll' in feat_name or 'vel' in feat_name
    marker = " [TEMPORAL]" if is_temporal else ""
    print(f"{rank:<5} {feat_name:<30} {importance:.6f}{marker}")

temporal_count = sum(1 for idx in indices if 'delta' in feature_cols[idx] or 't-' in feature_cols[idx] 
                     or 'roll' in feature_cols[idx] or 'vel' in feature_cols[idx])
print(f"\nTemporal features in top 15: {temporal_count}/15")


TOP 15 MOST IMPORTANT FEATURES:

Rank  Feature Name                     Importance
-------------------------------------------------------
1     file_id                        6884.000000
2     vel_y                          1373.000000 [TEMPORAL]
3     vel_z                          1207.000000 [TEMPORAL]
4     vel_x                          944.000000 [TEMPORAL]
5     mean_y_t-3                     436.000000 [TEMPORAL]
6     energy                         413.000000
7     mean_y                         411.000000
8     roll_mean_accel_5              391.000000 [TEMPORAL]
9     index                          380.000000
10    std_x_t-3                      367.000000 [TEMPORAL]
11    std_y_t-3                      346.000000 [TEMPORAL]
12    velocity_magnitude             330.000000 [TEMPORAL]
13    roll_std_accel_5               321.000000 [TEMPORAL]
14    mean_x                         308.000000
15    mean_z                         304.000000

Temporal features in top 15: 9/15


## 9. Generate Submission Predictions

In [20]:
# Generate predictions for all test files in submission format
print("Generating predictions for entire test set...\n")

submission_ids = []
submission_labels = []
pred_count = 0
pred_errors = 0

# Process all test files (with sampling for efficiency)
all_test_files = list(range(TRAIN_FILE_NUMBERS + 1, TRAIN_FILE_NUMBERS + TEST_FILE_NUMBERS + 1, 1))

for file_id in all_test_files:
    try:
        df = file_csv_to_df(False, file_id)
        df = add_temporal_features(df)
        df = add_sequence_context(df, history_steps=3)
        
        # Prepare features
        X_file = df[feature_cols].values
        X_file_scaled = scaler.transform(X_file)
        
        # Make predictions (use majority vote of all rows in file)
        preds = model.predict(X_file_scaled)
        majority_label = np.bincount(preds).argmax()
        
        submission_ids.append(file_id)
        submission_labels.append(majority_label)
        pred_count += 1
        
        if pred_count % 100 == 0:
            print(f"  Processed {pred_count} files...")
    except Exception as e:
        pred_errors += 1
        if pred_errors <= 3:
            print(f"Error on file {file_id}: {type(e).__name__}")
        # Use default label if error
        submission_ids.append(file_id)
        submission_labels.append(0)
        continue

print(f"\nGenerated predictions for {pred_count} files")
print(f"Errors encountered: {pred_errors}")

# Create submission dataframe
submission_df = pd.DataFrame({
    'Id': submission_ids,
    'Label': submission_labels
})

# Save to CSV
output_file = 'predictions_temporal_features.csv'
submission_df.to_csv(output_file, index=False)
print(f"\nSubmission saved to: {output_file}")
print(f"Format: Id, Label")
print(f"\nFirst 10 predictions:")
print(submission_df.head(10))

# Show label distribution
print(f"\nPrediction distribution:")
for label in range(6):
    count = np.sum(submission_df['Label'].values == label)
    pct = 100 * count / len(submission_df)
    print(f"  Label {label}: {count} files ({pct:.1f}%)")

Generating predictions for entire test set...

  Processed 100 files...
  Processed 200 files...
  Processed 300 files...
  Processed 400 files...
  Processed 500 files...
  Processed 600 files...
  Processed 700 files...
  Processed 800 files...
  Processed 900 files...
  Processed 1000 files...
  Processed 1100 files...
  Processed 1200 files...
  Processed 1300 files...
  Processed 1400 files...
  Processed 1500 files...
  Processed 1600 files...
  Processed 1700 files...
  Processed 1800 files...
  Processed 1900 files...
  Processed 2000 files...
  Processed 2100 files...
  Processed 2200 files...
  Processed 2300 files...
  Processed 2400 files...
  Processed 2500 files...
  Processed 2600 files...
  Processed 2700 files...
  Processed 2800 files...
  Processed 2900 files...
  Processed 3000 files...
  Processed 3100 files...
  Processed 3200 files...
  Processed 3300 files...
  Processed 3400 files...
  Processed 3500 files...
  Processed 3600 files...
  Processed 3700 files...


## 10. Summary

### How Sequential Accelerometer Readings are Aligned with Activity Labels:

**1. Temporal Feature Engineering**: 
- Each time series converted to feature vectors with temporal context
- First-order and second-order differences capture motion dynamics
- Rolling statistics preserve temporal patterns

**2. Sequential Context**:
- Previous timesteps (t-1, t-2, t-3) included as features
- Allows model to learn temporal dependencies without RNNs
- Cumulative velocity captures long-term motion patterns

**3. Temporal Dependencies Capture**:
- Gradient Boosting learns which temporal features best discriminate activities
- Delta features help identify activity transitions
- Rolling statistics capture activity-specific motion patterns